# Bài thực hành: Phép cuộn (Convolution) trên ảnh

## 1. Mục tiêu

- Hiểu và cài đặt phép cuộn 2D giữa ảnh và mẫu (kernel/mặt nạ).
- Quan sát tác dụng của các mẫu khác nhau: làm trơn (lọc trung bình) và làm sắc nét.
- Áp dụng phép cuộn cho ảnh xám và ảnh màu (xử lý từng kênh).

## 2. Đề bài

Cho một ảnh bất kỳ. Đọc ảnh dưới dạng **ảnh xám** và thực hiện phép cuộn với các mẫu sau:

**a. Mẫu trung bình (làm trơn):** mẫu là ảnh kích thước `(2n+1) × (2n+1)`, mọi phần tử đều bằng `1 / ((2n+1)*(2n+1))`. Thử nghiệm với `n = 1, 3, 5, 7, 9`.

**b. Mẫu làm sắc nét:**

```
T1 (3x3):                    T2 (5x5):
-1/8  -1/8  -1/8             -1/24 -1/24 -1/24 -1/24 -1/24
-1/8   2    -1/8             -1/24 -1/24 -1/24 -1/24 -1/24
-1/8  -1/8  -1/8             -1/24 -1/24   2   -1/24 -1/24
                             -1/24 -1/24 -1/24 -1/24 -1/24
                             -1/24 -1/24 -1/24 -1/24 -1/24
```

**c. Ảnh màu:** áp dụng tương tự cho ảnh màu, thực hiện phép cuộn trên từng kênh rồi ghép lại.

## 3. Yêu cầu

1. Tự cài đặt hàm phép cuộn 2D; có thể dùng hàm thư viện để đối chiếu.
2. Ảnh kết quả cùng kích thước ảnh gốc (dùng **điểm chốt** ở tâm mẫu).
3. Đưa kết quả về khoảng `[0, 255]` và kiểu `uint8` trước khi hiển thị.
4. Hiển thị ảnh gốc và ảnh kết quả cạnh nhau.
5. Nhận xét: ảnh thay đổi thế nào khi `n` tăng; tổng phần tử của T1, T2 và ảnh hưởng đến độ sáng; so sánh T1 với T2.

# Chuẩn bị
Ta tải các thư viện hỗ trợ vào và download ảnh ví dụ

In [ ]:
# Nếu chưa có thư viện, bỏ dấu # ở dòng dưới rồi chạy:
# %pip install opencv-python numpy

import os
import urllib.request
import cv2 # Tải thư viện OpenCV
import numpy as np # Tải thư viện Numpy
from IPython.display import display, Image # Hiển thị ảnh trong Jupyter

# Thay cho cv2_imshow của Colab: hiển thị ảnh (BGR hoặc xám) ngay trong ô kết quả
def cv2_imshow(anh):
    ok, buf = cv2.imencode(".png", anh)
    display(Image(data=buf.tobytes()))

# Đường dẫn ảnh: đổi thành ảnh của bạn nếu muốn (để cùng thư mục với notebook)
DUONG_DAN_ANH = "lena.jpg"
if not os.path.exists(DUONG_DAN_ANH):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg",
        DUONG_DAN_ANH)

# Hàm phép cuộn

Ta dùng lại công thức phép cuộn (có điểm chốt `p`) như bài mẫu: với mẫu `T` kích thước `m x n` (rộng x cao), điểm chốt `p = (px, py)`

$$I_{kq}(x, y) = \sum_{j=0}^{n-1}\sum_{i=0}^{m-1} I(x+i-p_x,\; y+j-p_y)\cdot T(i, j)$$

Các vị trí mà mẫu đặt lên bị tràn ra ngoài ảnh không tính được, ta gán bằng giá trị `inv`.

**Hàm 1 - `phep_cuon_2`:** cài đặt 4 vòng lặp đúng theo công thức (dễ hiểu, dùng để kiểm chứng).

In [ ]:
# M, N: kích thước ảnh; m, n: kích thước mẫu; p: điểm chốt
def co_the_tinh(x, y, M, N, m, n, p):
    x_min = x-p[0]
    y_min = y-p[1]
    x_max = x+m-p[0]-1
    y_max = y+n-p[1]-1
    return x_min>=0 and y_min>=0 and x_max<M and y_max<N

def phep_cuon_2(I, T, p=(0,0), inv=np.inf):
    M, N = I.shape[1], I.shape[0]
    m, n = T.shape[1], T.shape[0]
    I_kq = np.zeros((N, M), np.float32)
    for x in range(I_kq.shape[1]):
        for y in range(I_kq.shape[0]):
            if co_the_tinh(x, y, M, N, m, n, p):
                sum = 0.
                for j in range(n):
                    for i in range(m):
                        sum = sum + I[y+j-p[1],x+i-p[0]]*T[j, i]
                I_kq[y, x] = sum
            else:
                I_kq[y, x] = inv
    return I_kq

**Hàm 2 - `phep_cuon_nhanh`:** ảnh Lena có kích thước 512x512, với mẫu 19x19 thì hàm 1 phải thực hiện khoảng 95 triệu phép nhân bằng vòng lặp Python (rất chậm). Vì vậy ta viết hàm thứ hai cho kết quả **giống hệt** nhưng thay hai vòng lặp trên ảnh bằng phép cộng trên cả mảng: với mỗi phần tử `T[j, i]`, ta nhân nó với toàn bộ ảnh đã dịch đi `(i, j)` rồi cộng dồn.

In [ ]:
def phep_cuon_nhanh(I, T, p=(0,0), inv=np.inf):
    M, N = I.shape[1], I.shape[0]
    m, n = T.shape[1], T.shape[0]
    x0, x1 = p[0], M-m+p[0]+1   # vùng x tính được
    y0, y1 = p[1], N-n+p[1]+1   # vùng y tính được
    I_f = I.astype(np.float64)
    acc = np.zeros((y1-y0, x1-x0), np.float64)
    for j in range(n):
        for i in range(m):
            acc += I_f[j:j+y1-y0, i:i+x1-x0] * T[j, i]
    I_kq = np.full((N, M), inv, np.float32)
    I_kq[y0:y1, x0:x1] = acc
    return I_kq

Kiểm tra hai hàm cho cùng kết quả trên ảnh nhỏ (ảnh ví dụ trong bài mẫu), mẫu 3x3 với điểm chốt ở tâm `(1, 1)`:

In [ ]:
I = np.array([
    [1, 2, 4, 5, 8, 7],
    [2, 1, 1, 4, 2, 2],
    [4, 5, 5, 8, 8, 2],
    [1, 2, 1, 1, 4, 4],
    [7, 2, 2, 1, 5, 2]
], dtype=np.uint8)
T = np.full((3, 3), 1/9, dtype=np.float32)

kq_cham = phep_cuon_2(I, T, (1, 1), inv=0)
kq_nhanh = phep_cuon_nhanh(I, T, (1, 1), inv=0)
print(np.round(kq_nhanh, 2))
print("Hai hàm giống nhau:", np.allclose(kq_cham, kq_nhanh, atol=1e-4))

# Các hàm hỗ trợ hiển thị

Ảnh kết quả là ảnh số thực, và các mẫu làm sắc nét có hệ số âm nên giá trị có thể **âm hoặc lớn hơn 255**. Nếu ép kiểu `astype(np.uint8)` trực tiếp, giá trị sẽ bị "quay vòng" (ví dụ 260 thành 4, -3 thành 253) làm ảnh sai. Vì vậy ta phải **cắt ngưỡng về [0, 255]** trước khi ép kiểu.

Hàm `ghep_anh` ghép nhiều ảnh thành một hàng, thu nhỏ về cùng kích thước và ghi tiêu đề lên ảnh để dễ so sánh.

In [ ]:
def ve_uint8(I_kq):
    # cắt ngưỡng về [0, 255] rồi mới ép kiểu
    return np.clip(I_kq, 0, 255).astype(np.uint8)

def ghep_anh(ds_anh, ds_ten, kich_thuoc=256):
    ds = []
    for anh, ten in zip(ds_anh, ds_ten):
        a = cv2.resize(anh, (kich_thuoc, kich_thuoc))
        if a.ndim == 2:
            a = cv2.cvtColor(a, cv2.COLOR_GRAY2BGR)
        cv2.putText(a, ten, (5, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2)
        ds.append(a)
    return np.hstack(ds)

# Đọc ảnh xám
Ta đọc file dạng ảnh xám:

In [ ]:
I_xam = cv2.imread(DUONG_DAN_ANH, cv2.IMREAD_GRAYSCALE)
print("Kích thước ảnh xám:", I_xam.shape)
cv2_imshow(I_xam)

# Phần a. Mẫu trung bình (làm trơn)

Với `n` cho trước, mẫu có kích thước `(2n+1) x (2n+1)`, mọi phần tử bằng `1/(2n+1)^2` (tổng bằng 1). Điểm chốt đặt ở tâm mẫu `(n, n)` để ảnh kết quả không bị lệch. Vùng viền không tính được ta gán 0 (màu đen).

In [ ]:
def mau_trung_binh(n):
    k = 2*n + 1
    return np.full((k, k), 1.0/(k*k), dtype=np.float32)

ds_n = [1, 3, 5, 7, 9]
kq_tb = {}
for n in ds_n:
    T_tb = mau_trung_binh(n)
    kq_tb[n] = phep_cuon_nhanh(I_xam, T_tb, p=(n, n), inv=0)
    print(f"n={n}: mẫu {T_tb.shape[0]}x{T_tb.shape[1]}, mỗi phần tử = {T_tb[0,0]:.5f}, tổng = {T_tb.sum():.2f}")

In [ ]:
# Hàng 1: ảnh gốc + n=1, 3   |   Hàng 2: n=5, 7, 9
hang1 = ghep_anh([I_xam, ve_uint8(kq_tb[1]), ve_uint8(kq_tb[3])],
                 ["Anh goc", "n=1 (3x3)", "n=3 (7x7)"])
hang2 = ghep_anh([ve_uint8(kq_tb[5]), ve_uint8(kq_tb[7]), ve_uint8(kq_tb[9])],
                 ["n=5 (11x11)", "n=7 (15x15)", "n=9 (19x19)"])
cv2_imshow(np.vstack([hang1, hang2]))

Để có bằng chứng định lượng cho nhận xét, ta đo **độ nét** của ảnh bằng phương sai của ảnh Laplace (giá trị càng lớn thì ảnh càng nhiều chi tiết/cạnh rõ). Ta chỉ đo ở vùng lõi 20 điểm ảnh cách viền để không bị ảnh hưởng bởi viền đen.

In [ ]:
def do_net(I_kq, le=20):
    vung = I_kq[le:-le, le:-le].astype(np.float64)
    return cv2.Laplacian(vung, cv2.CV_64F).var()

def do_sang(I_kq, le=20):
    return I_kq[le:-le, le:-le].mean()

print(f"{'Anh':<14}{'Độ sáng TB':>12}{'Độ nét':>12}")
print(f"{'Ảnh gốc':<14}{do_sang(I_xam):>12.2f}{do_net(I_xam):>12.2f}")
for n in ds_n:
    print(f"{'n='+str(n):<14}{do_sang(kq_tb[n]):>12.2f}{do_net(kq_tb[n]):>12.2f}")

### Nhận xét phần a

- Khi `n` tăng, ảnh **càng mờ**: mỗi điểm ảnh mới là trung bình của một vùng lân cận `(2n+1) x (2n+1)` càng rộng, nên các chi tiết nhỏ, cạnh và nhiễu bị trung hoà, chỉ còn lại các mảng sáng tối lớn. Độ nét (phương sai Laplace) giảm mạnh theo `n`.
- Độ sáng trung bình gần như không đổi, vì tổng các phần tử của mẫu bằng 1.
- Khi `n` tăng, **vùng viền đen** (không tính được) cũng dày lên: rộng đúng `n` điểm ảnh ở mỗi phía, và khối lượng tính toán tăng theo `(2n+1)^2`.

# Phần b. Mẫu làm sắc nét T1, T2

Cả hai mẫu đều có phần tử tâm bằng 2 và các phần tử xung quanh bằng `-1/8` (T1, 8 phần tử) hoặc `-1/24` (T2, 24 phần tử). Ta kiểm tra tổng các phần tử:

In [ ]:
T1 = np.full((3, 3), -1/8, dtype=np.float32)
T1[1, 1] = 2

T2 = np.full((5, 5), -1/24, dtype=np.float32)
T2[2, 2] = 2

print("T1 =\n", T1)
print("Tổng T1 =", T1.sum())
print("\nT2 =\n", T2)
print("Tổng T2 =", T2.sum())

In [ ]:
kq_T1 = phep_cuon_nhanh(I_xam, T1, p=(1, 1), inv=0)
kq_T2 = phep_cuon_nhanh(I_xam, T2, p=(2, 2), inv=0)

cv2_imshow(ghep_anh([I_xam, ve_uint8(kq_T1), ve_uint8(kq_T2)],
                    ["Anh goc", "T1 (3x3)", "T2 (5x5)"], kich_thuoc=400))

print(f"{'Anh':<14}{'Độ sáng TB':>12}{'Độ nét':>12}")
print(f"{'Ảnh gốc':<14}{do_sang(I_xam):>12.2f}{do_net(I_xam):>12.2f}")
print(f"{'T1':<14}{do_sang(kq_T1):>12.2f}{do_net(kq_T1):>12.2f}")
print(f"{'T2':<14}{do_sang(kq_T2):>12.2f}{do_net(kq_T2):>12.2f}")

Nếu **không** cắt ngưỡng mà ép kiểu trực tiếp thì kết quả sai như sau (các điểm có giá trị âm hoặc lớn hơn 255 bị quay vòng, xuất hiện các đốm đen/trắng bất thường):

In [ ]:
sai = kq_T1.astype(np.uint8)   # ép kiểu trực tiếp, không cắt ngưỡng
cv2_imshow(ghep_anh([ve_uint8(kq_T1), sai], ["Cat nguong [0,255]", "Ep kieu truc tiep"], kich_thuoc=400))
print("Giá trị nhỏ nhất / lớn nhất của kết quả T1:", kq_T1[2:-2, 2:-2].min(), "/", kq_T1[2:-2, 2:-2].max())

### Nhận xét phần b

- Tổng các phần tử: **T1 = 2 - 8 x (1/8) = 1**, **T2 = 2 - 24 x (1/24) = 1**. Vì tổng bằng 1 nên ở vùng ảnh phẳng (các điểm lân cận có giá trị bằng nhau) kết quả đúng bằng giá trị gốc: **độ sáng trung bình được giữ nguyên**. Nếu tổng lớn hơn 1 ảnh sẽ sáng lên, nhỏ hơn 1 ảnh sẽ tối đi, còn bằng 0 thì chỉ còn lại cạnh trên nền đen.
- Về bản chất, T1 và T2 = (điểm tâm x 2) - (trung bình các điểm lân cận) = ảnh gốc + (ảnh gốc - ảnh làm trơn). Phần "ảnh gốc - ảnh làm trơn" chính là thành phần chi tiết/cạnh, được cộng thêm vào ảnh gốc nên ảnh **sắc nét hơn**, độ nét đo được tăng so với ảnh gốc.
- **So sánh T1 và T2:** T1 chỉ so sánh với 8 điểm lân cận sát tâm nên làm nổi các chi tiết rất nhỏ, mịn (và cả nhiễu hạt). T2 so sánh điểm ảnh với vùng lân cận 5x5 rộng hơn nên làm nổi cả các chi tiết/cạnh lớn hơn, hiệu ứng viền sáng-tối quanh cạnh rõ và "gắt" hơn, tương phản cục bộ tăng mạnh hơn; đổi lại nhiễu cũng bị khuếch đại nhiều hơn và vùng viền đen dày hơn (2 điểm so với 1 điểm).
- Do có hệ số âm, kết quả có thể ra ngoài [0, 255], nên bắt buộc cắt ngưỡng trước khi ép kiểu `uint8`.

# Phần c. Ảnh màu (xử lý từng kênh)

Ảnh màu đọc bằng OpenCV có 3 kênh theo thứ tự B, G, R. Ta tách thành 3 ảnh xám, thực hiện phép cuộn **độc lập** trên từng kênh với cùng một mẫu, rồi ghép lại bằng `cv2.merge`.

In [ ]:
I_mau = cv2.imread(DUONG_DAN_ANH, cv2.IMREAD_COLOR)
print("Kích thước ảnh màu:", I_mau.shape)

def phep_cuon_mau(I_mau, T, p, inv=0):
    kenh = cv2.split(I_mau)  # B, G, R
    kq_kenh = [ve_uint8(phep_cuon_nhanh(k, T, p, inv)) for k in kenh]
    return cv2.merge(kq_kenh)

cv2_imshow(I_mau)

In [ ]:
# a. Mẫu trung bình
kq_mau_tb = {n: phep_cuon_mau(I_mau, mau_trung_binh(n), (n, n)) for n in ds_n}

hang1 = ghep_anh([I_mau, kq_mau_tb[1], kq_mau_tb[3]],
                 ["Anh goc", "n=1 (3x3)", "n=3 (7x7)"])
hang2 = ghep_anh([kq_mau_tb[5], kq_mau_tb[7], kq_mau_tb[9]],
                 ["n=5 (11x11)", "n=7 (15x15)", "n=9 (19x19)"])
cv2_imshow(np.vstack([hang1, hang2]))

In [ ]:
# b. Mẫu làm sắc nét
kq_mau_T1 = phep_cuon_mau(I_mau, T1, (1, 1))
kq_mau_T2 = phep_cuon_mau(I_mau, T2, (2, 2))

cv2_imshow(ghep_anh([I_mau, kq_mau_T1, kq_mau_T2],
                    ["Anh goc", "T1 (3x3)", "T2 (5x5)"], kich_thuoc=400))

## Đối chiếu với hàm thư viện OpenCV

`cv2.filter2D` cũng thực hiện đúng công thức trên (các mẫu của bài đều đối xứng nên phép cuộn và phép tương quan cho cùng kết quả). Ta so sánh trên vùng lõi (bỏ viền) để kiểm tra hàm tự cài đặt.

In [ ]:
def so_sanh(I, T, p):
    tu_cai = np.clip(phep_cuon_nhanh(I, T, p, inv=0), 0, 255)
    thu_vien = cv2.filter2D(I.astype(np.float32), cv2.CV_32F, T, borderType=cv2.BORDER_REPLICATE)
    thu_vien = np.clip(thu_vien, 0, 255)
    l = max(T.shape)             # bỏ viền
    return np.abs(tu_cai[l:-l, l:-l] - thu_vien[l:-l, l:-l]).max()

for n in ds_n:
    print(f"Trung bình n={n}: sai khác lớn nhất = {so_sanh(I_xam, mau_trung_binh(n), (n, n)):.4f}")
print(f"T1: sai khác lớn nhất = {so_sanh(I_xam, T1, (1, 1)):.4f}")
print(f"T2: sai khác lớn nhất = {so_sanh(I_xam, T2, (2, 2)):.4f}")

### Nhận xét phần c

- Xử lý từng kênh độc lập với cùng một mẫu cho kết quả tương tự ảnh xám: mẫu trung bình làm ảnh màu mờ dần khi `n` tăng, mẫu T1, T2 làm ảnh màu nét hơn. Màu sắc tổng thể được giữ nguyên vì tổng các phần tử của mẫu bằng 1 ở mọi kênh.
- Với mẫu làm sắc nét, vì mỗi kênh được làm nét riêng và bị cắt ngưỡng riêng, ở các cạnh có độ tương phản lớn có thể xuất hiện chút **viền màu** (sai lệch sắc độ) so với ảnh gốc.
- Kết quả hàm tự cài đặt trùng với `cv2.filter2D` (sai khác gần bằng 0, chỉ do làm tròn số thực), chứng tỏ cài đặt đúng.

# Kết luận

| Mẫu | Tổng phần tử | Tác dụng |
|---|---|---|
| Trung bình `(2n+1)x(2n+1)` | 1 | Làm trơn/làm mờ, `n` càng lớn càng mờ, viền đen càng dày |
| T1 (3x3) | 1 | Làm sắc nét, nổi chi tiết nhỏ |
| T2 (5x5) | 1 | Làm sắc nét mạnh hơn, nổi chi tiết lớn hơn, khuếch đại nhiễu nhiều hơn |

Các mẫu có tổng bằng 1 thì giữ nguyên độ sáng trung bình; mẫu có hệ số âm thì kết quả có thể ngoài [0, 255] nên phải cắt ngưỡng trước khi ép kiểu về `uint8`.